# Project Delphi (Merlin)

## 02 - Title Embeddings

### Overview:

In this notebook, we'll generate **semantic embeddings** for YouTube video titles using the `all-MiniLM-L6-v2` model from [Sentence Transformers](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2). These embeddings will capture the *contextual meaning* of each title -- allowing our downstream XGBoost models to learn patterns beyond just word counts or keywords.  

In other words, we're transforming human language into a numeric representation that the model can understand.

---

### The Plan:
1. **Load** the cleaned dataset from `01_data_preparation`  
2. **Initialize** the Sentence Transformer model (`all-MiniLM-L6-v2`)  
3. **Encode** all video titles into 384-dimensional embeddings  
4. **Append** these embeddings to the main DataFrame  
5. **Save** the new feature-rich dataset for the next stage (`03_model_training_xgboost`)

---

### Why This Matters

Embedding the titles gives our model a way to:
- Understand *semantic similarity* (e.g., “49ers lose tough game” ≈ “San Francisco falls short”)
- Capture *emotional tone* and *narrative framing*
- Move beyond naive keyword matching and learn richer title-performance relationships

## Load the cleaned (and engineered) dataset

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load the cleaned and engineered dataset
import pandas as pd

data_path = '/content/drive/MyDrive/Colab Notebooks/project_delphi/data/project_delphi_clean.pkl'
df = pd.read_pickle(data_path)

# Check the first few rows of the DataFrame
df.head()

,title,duration_seconds,day_of_week,month,views_lifetime,subs_lifetime
3,Radio Layoffs Hit KNBR (Again) - Damon & Sam B...,5306.0,3,10,10666.0,45.0
16,Analysis: 49ers Lose To Lions In Disastrous Fa...,5234.0,1,11,5617.0,5.0
19,"49ers Insider Matt Maiocco Talks Trades, QB Co...",4611.0,1,9,5246.0,13.0
21,"49ers Analysis: Campbell Is A LOSER, Purdy TER...",5875.0,4,11,4995.0,9.0
23,49ers Analysis: Embarrassing Late-Game Loss vs...,7007.0,0,11,4814.0,37.0


In [ ]:
# And confirm the shape -- should be (298, 6)
df.shape

(298, 6)

## Initialize the Sentence Transformer model

For this step, we'll use **`all-MiniLM-L6-v2`**, a lightweight yet powerful transformer that turns each video title into **a 384-dimensional embedding** capturing its meaning and tone.

In [ ]:
# Initialize the Sentence Transformer model
from sentence_transformers import SentenceTransformer

# Load the pre-trained model, as speficied above
model = SentenceTransformer('all-MiniLM-L6-v2')
model

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

The `all-MiniLM-L6-v2` model is a streamlined version of BERT that compresses its knowlegde into a smaller 6-layer transformer and then applies a **pooling layer** to combine word-level embeddings into a single 384-dimensional vector capturing the overall meaning, tone, and context of each title.

## Generate the title embeddings

We'll encode each video title into a 384-dimensional vector using the model we just loaded.

In [ ]:
# Start by importing numpy for computations
import numpy as np

# Optional check: make sure the column 'titles' actually exists
# assert 'title' in df.columns, "Expected a 'title' column in the dataframe."
# titles = df['title'].astype(str).fillna('').tolist()

# Generate 384-dimensional embeddings for each video title using the pre-trained Sentence Transformer
# Important notes:
# - titles: This is the list of text strings to embed (each representing a video title)
# - batch_size=64: Small batches improve efficiency and avoid memory overload
# - show_progress_bar=True: This shows a progress bar for visibility during encoding
# - convert_to_numpy=True: This returns the embeddings as a NumPy array (easy to use later with ML models)
# - normalize_embeddings=True: This scales each embedding vector to have the same size, helpful for comparison
embeddings = model.encode(titles, batch_size=64, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)

# Display the embeddings
embeddings

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

array([[-0.03242624,  0.00422182, -0.05105253, ..., -0.03286464,
        -0.09195392,  0.03611416],
       [-0.04651034,  0.02099138, -0.11906488, ...,  0.03547176,
         0.017452  ,  0.00791078],
       [-0.09219025, -0.0811388 , -0.09714816, ...,  0.0297546 ,
        -0.00691456,  0.0876679 ],
       ...,
       [-0.0151018 ,  0.01407246, -0.01868924, ..., -0.08094913,
         0.01787599, -0.01089243],
       [-0.01747648,  0.04252426, -0.05791993, ...,  0.02059736,
         0.03103362,  0.0579368 ],
       [-0.08775537,  0.05134858, -0.01422159, ...,  0.01069328,
        -0.05193787,  0.01165167]], dtype=float32)

In [ ]:
# Check the shape of the embeddings
# Note: This should be (298, 384) becasue we have 298 title embeddings that are 384-dimensional
embeddings.shape

(298, 384)

## Append the embeddings to the cleaned (and engineered) DataFrame

Now that we've generated the embeddings, we're ready to append them to our DataFrame (`df`). This way, every video title's semantic representation will be mapped onto its numeric features -- creating a unified dataset that blends the two types of input features.

In [ ]:
# Need to turn the embeddings into a DataFrame, as they are currently an array
# type(embeddings)

# Use the .DataFrame() function
# Note: The f-string + list comprehension create those column names we see below!
embeddings_df = pd.DataFrame(embeddings, columns=[f'embed_{i}' for i in range(embeddings.shape[1])])

# Check the first few rows
embeddings_df.head()

,embed_0,embed_1,embed_2,embed_3,embed_4,embed_5,embed_6,embed_7,embed_8,embed_9,...,embed_374,embed_375,embed_376,embed_377,embed_378,embed_379,embed_380,embed_381,embed_382,embed_383
0,-0.032426,0.004222,-0.051053,-0.048461,-0.025876,0.056063,0.008602,-0.018557,0.019126,0.013981,...,0.032988,-0.036669,-0.082085,0.019063,-0.062317,-0.000892,-0.088869,-0.032865,-0.091954,0.036114
1,-0.046510,0.020991,-0.119065,0.021533,0.050918,0.035446,-0.010613,0.050547,-0.005748,0.017869,...,0.056652,0.021922,-0.033295,-0.063782,-0.070290,0.008837,0.021913,0.035472,0.017452,0.007911
2,-0.092190,-0.081139,-0.097148,0.006893,0.001374,-0.005492,0.088969,0.029654,0.059486,0.043821,...,0.021139,0.039691,-0.008243,0.007969,-0.036037,0.009903,-0.057307,0.029755,-0.006915,0.087668
3,-0.069882,-0.004827,-0.073440,-0.000911,-0.036637,-0.006644,0.148387,0.024400,0.070921,-0.010281,...,0.084864,0.018908,-0.079154,-0.021822,-0.078934,0.033437,0.010721,-0.007301,0.048347,0.006759
4,-0.056615,0.003609,-0.061708,0.000720,0.058359,0.000997,0.039149,-0.005540,0.036926,0.003661,...,0.116987,-0.016363,-0.048928,-0.050522,-0.121011,0.020156,-0.032676,0.002244,-0.048972,0.029054


In [ ]:
# Now to combine or concatenate the two DataFrames
# Note: 1) .reset_index(drop=True) makes sure the two DataFrames match up row-by-row
# and 2) axix=1 tells pandas to add columns side-by-side instead of stacking rows
df_with_embeddings = pd.concat([df.reset_index(drop=True), embeddings_df], axis=1)

# Check the results -- or at least the first few rows
df_with_embeddings.head()

,title,duration_seconds,day_of_week,month,views_lifetime,subs_lifetime,embed_0,embed_1,embed_2,embed_3,...,embed_374,embed_375,embed_376,embed_377,embed_378,embed_379,embed_380,embed_381,embed_382,embed_383
0,Radio Layoffs Hit KNBR (Again) - Damon & Sam B...,5306.0,3,10,10666.0,45.0,-0.032426,0.004222,-0.051053,-0.048461,...,0.032988,-0.036669,-0.082085,0.019063,-0.062317,-0.000892,-0.088869,-0.032865,-0.091954,0.036114
1,Analysis: 49ers Lose To Lions In Disastrous Fa...,5234.0,1,11,5617.0,5.0,-0.046510,0.020991,-0.119065,0.021533,...,0.056652,0.021922,-0.033295,-0.063782,-0.070290,0.008837,0.021913,0.035472,0.017452,0.007911
2,"49ers Insider Matt Maiocco Talks Trades, QB Co...",4611.0,1,9,5246.0,13.0,-0.092190,-0.081139,-0.097148,0.006893,...,0.021139,0.039691,-0.008243,0.007969,-0.036037,0.009903,-0.057307,0.029755,-0.006915,0.087668
3,"49ers Analysis: Campbell Is A LOSER, Purdy TER...",5875.0,4,11,4995.0,9.0,-0.069882,-0.004827,-0.073440,-0.000911,...,0.084864,0.018908,-0.079154,-0.021822,-0.078934,0.033437,0.010721,-0.007301,0.048347,0.006759
4,49ers Analysis: Embarrassing Late-Game Loss vs...,7007.0,0,11,4814.0,37.0,-0.056615,0.003609,-0.061708,0.000720,...,0.116987,-0.016363,-0.048928,-0.050522,-0.121011,0.020156,-0.032676,0.002244,-0.048972,0.029054


In [ ]:
# And now check the shape of the new DataFrame
df_with_embeddings.shape

(298, 390)

The above shape tells us we have 298 rows (videos) and 390 coluns (4 numeric features, 384 embedding dimensions and 2 target variables). That means we're ready to save the DataFrame (`df_with_embeddings`) and dive into the modeling.

## Save the feature-rich dataset (`df_with_embeddings`)

In [ ]:
# Save the feature-rich dataset for modeling
df_with_embeddings.to_pickle('/content/drive/MyDrive/Colab Notebooks/project_delphi/data/project_delphi_with_embeddings.pkl')
df_with_embeddings.to_csv('/content/drive/MyDrive/Colab Notebooks/project_delphi/data/project_delphi_with_embeddings.csv', index=False)


In [2]:
import sentence_transformers
sentence_transformers.__version__

'5.1.2'

In [3]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

embedder.save("/content/merlin_embedder")
!zip -r /content/merlin_embedder.zip /content/merlin_embedder

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  adding: content/merlin_embedder/ (stored 0%)
  adding: content/merlin_embedder/vocab.txt (deflated 53%)
  adding: content/merlin_embedder/sentence_bert_config.json (deflated 9%)
  adding: content/merlin_embedder/README.md (deflated 64%)
  adding: content/merlin_embedder/config.json (deflated 47%)
  adding: content/merlin_embedder/2_Normalize/ (stored 0%)
  adding: content/merlin_embedder/config_sentence_transformers.json (deflated 40%)
  adding: content/merlin_embedder/tokenizer_config.json (deflated 73%)
  adding: content/merlin_embedder/model.safetensors (deflated 9%)
  adding: content/merlin_embedder/special_tokens_map.json (deflated 80%)
  adding: content/merlin_embedder/1_Pooling/ (stored 0%)
  adding: content/merlin_embedder/1_Pooling/config.json (deflated 59%)
  adding: content/merlin_embedder/modules.json (deflated 62%)
  adding: content/merlin_embedder/tokenizer.json (deflated 71%)
